In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import warnings
warnings.filterwarnings("ignore")

print("LangSmith Key:", "✅" if os.getenv("LANGCHAIN_API_KEY") else "❌ 未设置")
print("OpenAI Key:",    "✅" if os.getenv("OPENAI_API_KEY") else "❌ 未设置")
print("Tracing:",       os.getenv("LANGCHAIN_TRACING_V2"))
print("Project:",       os.getenv("LANGCHAIN_PROJECT"))

LangSmith Key: ✅
OpenAI Key: ✅
Tracing: true
Project: ai-investment-agent


In [ ]:
import pandas as pd
import numpy as np
import joblib
import yfinance as yf
import time
from typing import TypedDict, Annotated
import operator

# LangChain / LangGraph
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# 加载模型缓存
BEST_MODEL  = joblib.load("data/models/best_model.pkl")
BEST_NAME   = joblib.load("data/models/best_model_name.pkl")
EXPLAINER   = joblib.load("data/models/shap/shap_explainer.pkl")
SHAP_IMP    = pd.read_csv("data/models/shap/shap_importance.csv")
SCHEMA      = joblib.load("data/ml/schema.pkl")
SCALER      = joblib.load("data/ml/scaler.pkl")
X_TEST      = pd.read_csv("data/ml/X_test.csv", index_col=0, parse_dates=True)
META        = pd.read_csv("data/ml/test_meta.csv", index_col=0, parse_dates=True)

# LLM 初始化
llm = ChatOpenAI(model="gpt-5.5", temperature=0)

print(f"模型加载完成: {BEST_NAME}")
print(f"LLM: GPT-4o")

模型加载完成: XGBoost
LLM: GPT-4o


In [3]:
@tool
def PriceTool(ticker: str) -> dict:
    """
    获取指定股票的最新价格数据和技术指标。
    返回收盘价、RSI、MACD、布林带等技术指标。
    
    Args:
        ticker: 股票代码，如 NVDA、AAPL、TSLA
    """
    try:
        stock = yf.Ticker(ticker)
        hist  = stock.history(period="3mo")
        
        if len(hist) < 30:
            return {"error": f"Insufficient data for {ticker}"}
        
        close  = hist["Close"]
        volume = hist["Volume"]
        
        # RSI
        delta = close.diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        rs    = gain / loss
        rsi   = (100 - 100 / (1 + rs)).iloc[-1]
        
        # MACD
        ema12 = close.ewm(span=12).mean()
        ema26 = close.ewm(span=26).mean()
        macd  = (ema12 - ema26).iloc[-1]
        
        # Bollinger Bands
        sma20 = close.rolling(20).mean()
        std20 = close.rolling(20).std()
        bb_upper = (sma20 + 2 * std20).iloc[-1]
        bb_lower = (sma20 - 2 * std20).iloc[-1]
        bb_width = ((bb_upper - bb_lower) / sma20.iloc[-1])
        
        # 近期收益率
        ret_1d = close.pct_change(1).iloc[-1]
        ret_5d = close.pct_change(5).iloc[-1]
        
        return {
            "ticker":       ticker,
            "current_price": round(float(close.iloc[-1]), 2),
            "rsi_14":        round(float(rsi), 2),
            "macd":          round(float(macd), 4),
            "bb_upper":      round(float(bb_upper), 2),
            "bb_lower":      round(float(bb_lower), 2),
            "bb_width":      round(float(bb_width), 4),
            "return_1d":     round(float(ret_1d), 4),
            "return_5d":     round(float(ret_5d), 4),
            "volume_latest": int(volume.iloc[-1]),
            "signal_hint":   "OVERBOUGHT" if rsi > 70 else "OVERSOLD" if rsi < 30 else "NEUTRAL"
        }
    except Exception as e:
        return {"error": str(e), "ticker": ticker}

In [4]:
@tool
def SentimentTool(ticker: str) -> dict:
    """
    获取指定股票的新闻情感分析结果。
    返回情感分数、正负面比例和样本标题。
    
    Args:
        ticker: 股票代码，如 NVDA、AAPL、TSLA
    """
    try:
        sent_path = "data/sentiment/news_with_sentiment.csv"
        df = pd.read_csv(sent_path)
        df_t = df[df["ticker"] == ticker]
        
        if len(df_t) == 0:
            return {"error": f"No sentiment data for {ticker}"}
        
        mean_score = float(df_t["sentiment_numeric"].mean())
        pos_ratio  = float((df_t["sentiment_label"] == "positive").mean())
        neg_ratio  = float((df_t["sentiment_label"] == "negative").mean())
        
        signal = ("BULLISH" if mean_score > 0.1
                  else "BEARISH" if mean_score < -0.1
                  else "NEUTRAL")
        
        return {
            "ticker":              ticker,
            "total_articles":      len(df_t),
            "avg_sentiment_score": round(mean_score, 3),
            "positive_ratio":      round(pos_ratio, 3),
            "negative_ratio":      round(neg_ratio, 3),
            "overall_signal":      signal,
            "sample_headlines":    df_t["headline"].head(3).tolist()
        }
    except Exception as e:
        return {"error": str(e), "ticker": ticker}

In [5]:
@tool
def QuantTool(ticker: str) -> dict:
    """
    使用训练好的 XGBoost 模型预测指定股票明日涨跌方向。
    返回预测方向、置信区间和 SHAP 特征重要性解释。
    
    Args:
        ticker: 股票代码，如 NVDA、AAPL、TSLA
    """
    try:
        mask    = META["ticker"].values == ticker
        indices = np.where(mask)[0]
        
        if len(indices) == 0:
            return {"error": f"No test data for {ticker}"}
        
        X_input = X_TEST.iloc[indices].tail(1)
        pred    = int(BEST_MODEL.predict(X_input)[0])
        prob    = float(BEST_MODEL.predict_proba(X_input)[0][1])
        
        top_features = SHAP_IMP.head(5)[["feature", "mean_abs_shap"]].to_dict("records")
        
        return {
            "ticker":            ticker,
            "model":             BEST_NAME,
            "prediction":        "UP" if pred == 1 else "DOWN",
            "up_probability":    round(prob, 3),
            "confidence_interval": [
                round(max(0, prob - 0.1), 3),
                round(min(1, prob + 0.1), 3)
            ],
            "signal":            ("BUY"  if prob > 0.55
                                  else "SELL" if prob < 0.45
                                  else "HOLD"),
            "top_shap_features": top_features
        }
    except Exception as e:
        return {"error": str(e), "ticker": ticker}

In [6]:
pip_result = os.system(
    "pip install sentence-transformers chromadb -i "
    "https://mirrors.aliyun.com/pypi/simple/ -q"
)
@tool
def RAGTool(query: str, ticker: str = None) -> dict:
    """
    基于语义搜索从新闻数据库中检索最相关的新闻。
    用于为投资决策提供新闻证据支撑。
    
    Args:
        query:  搜索查询，如 "NVDA earnings growth AI chip demand"
        ticker: 可选，限定搜索范围到特定股票
    """
    try:
        import chromadb
        from sentence_transformers import SentenceTransformer
        
        # 加载或创建向量库
        chroma_path = "data/rag/chroma_db"
        os.makedirs(chroma_path, exist_ok=True)
        
        client     = chromadb.PersistentClient(path=chroma_path)
        model_name = "all-MiniLM-L6-v2"
        
        # 检查集合是否存在
        existing = [c.name for c in client.list_collections()]
        
        if "news_headlines" not in existing:
            # 首次运行：构建向量库
            print("首次运行，构建向量库...")
            encoder    = SentenceTransformer(model_name)
            collection = client.create_collection("news_headlines")
            
            df_news = pd.read_csv("data/sentiment/news_with_sentiment.csv")
            df_news = df_news.dropna(subset=["headline"])
            
            headlines = df_news["headline"].tolist()
            tickers_  = df_news["ticker"].tolist()
            sentiments = df_news["sentiment_label"].tolist()
            
            # 批量 embed
            batch_size = 64
            embeddings = []
            for i in range(0, len(headlines), batch_size):
                batch = headlines[i:i+batch_size]
                embs  = encoder.encode(batch, show_progress_bar=False)
                embeddings.extend(embs.tolist())
            
            ids = [f"doc_{i}" for i in range(len(headlines))]
            metadatas = [{"ticker": t, "sentiment": s}
                         for t, s in zip(tickers_, sentiments)]
            
            collection.add(
                embeddings=embeddings,
                documents=headlines,
                metadatas=metadatas,
                ids=ids
            )
            print(f"向量库构建完成，共 {len(headlines)} 条文档")
        else:
            encoder    = SentenceTransformer(model_name)
            collection = client.get_collection("news_headlines")
        
        # 语义检索
        query_emb = encoder.encode([query]).tolist()
        
        where_filter = {"ticker": ticker} if ticker else None
        results = collection.query(
            query_embeddings=query_emb,
            n_results=5,
            where=where_filter
        )
        
        docs      = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]
        
        retrieved = []
        for doc, meta, dist in zip(docs, metadatas, distances):
            retrieved.append({
                "headline":  doc,
                "ticker":    meta.get("ticker"),
                "sentiment": meta.get("sentiment"),
                "relevance": round(1 - dist, 3)
            })
        
        return {
            "query":            query,
            "ticker_filter":    ticker,
            "retrieved_count":  len(retrieved),
            "results":          retrieved
        }
    except Exception as e:
        return {"error": str(e), "query": query}

In [7]:
import sqlite3
import json
from datetime import datetime

class MemoryStore:
    """持久化推荐历史，支持多轮对话查询"""
    
    def __init__(self, db_path="data/memory/agent_memory.db"):
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        self.db_path = db_path
        self._init_db()
    
    def _init_db(self):
        conn = sqlite3.connect(self.db_path)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS recommendations (
                id        INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT,
                ticker    TEXT,
                signal    TEXT,
                reasoning TEXT,
                tool_data TEXT
            )
        """)
        conn.commit()
        conn.close()
    
    def save(self, ticker: str, signal: str, reasoning: str, tool_data: dict):
        conn = sqlite3.connect(self.db_path)
        conn.execute(
            "INSERT INTO recommendations (timestamp, ticker, signal, reasoning, tool_data) "
            "VALUES (?, ?, ?, ?, ?)",
            (datetime.now().isoformat(), ticker, signal,
             reasoning, json.dumps(tool_data))
        )
        conn.commit()
        conn.close()
    
    def get_history(self, ticker: str = None, limit: int = 5) -> list:
        conn   = sqlite3.connect(self.db_path)
        if ticker:
            rows = conn.execute(
                "SELECT timestamp, ticker, signal, reasoning FROM recommendations "
                "WHERE ticker=? ORDER BY timestamp DESC LIMIT ?",
                (ticker, limit)
            ).fetchall()
        else:
            rows = conn.execute(
                "SELECT timestamp, ticker, signal, reasoning FROM recommendations "
                "ORDER BY timestamp DESC LIMIT ?",
                (limit,)
            ).fetchall()
        conn.close()
        return [{"timestamp": r[0], "ticker": r[1],
                 "signal": r[2], "reasoning": r[3]} for r in rows]

memory = MemoryStore()
print("MemoryStore 初始化完成")

MemoryStore 初始化完成


In [8]:
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator

# Agent State 定义
class AgentState(TypedDict):
    messages:     Annotated[Sequence[BaseMessage], operator.add]
    ticker:       str
    tool_results: dict
    final_signal: str
    iterations:   int

# 工具列表
tools = [PriceTool, SentimentTool, QuantTool, RAGTool]

# LLM 绑定工具
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are an expert AI Investment Research Agent.

When analyzing a stock, you MUST call these tools IN ORDER:
1. PriceTool - get current price and technical indicators
2. SentimentTool - get news sentiment analysis  
3. QuantTool - get ML model prediction with SHAP explanation
4. RAGTool - retrieve relevant news evidence

After calling all tools, synthesize the results and provide:
- Final signal: BUY / SELL / HOLD
- Confidence level: HIGH / MEDIUM / LOW
- Key reasoning: 2-3 sentences explaining the decision
- Risk factors: 1-2 key risks to watch

Be concise and data-driven. Always cite specific numbers from tool outputs."""

def agent_node(state: AgentState):
    """主推理节点"""
    messages = state["messages"]
    
    # 加入系统提示
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(messages)
    
    response = llm_with_tools.invoke(messages)
    
    return {
        "messages":   [response],
        "iterations": state.get("iterations", 0) + 1
    }

def should_continue(state: AgentState):
    """ReAct 路由：有工具调用继续，否则结束"""
    last_message = state["messages"][-1]
    
    # 错误回退：超过8轮强制结束
    if state.get("iterations", 0) >= 8:
        return "end"
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "end"

# 构建 StateGraph
tool_node = ToolNode(tools)

graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")
graph.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "end": END}
)
graph.add_edge("tools", "agent")

agent_app = graph.compile()
print("LangGraph Agent 构建完成")
print("节点: agent → tools → agent (ReAct loop) → END")

LangGraph Agent 构建完成
节点: agent → tools → agent (ReAct loop) → END


In [9]:
def run_agent(ticker: str, query: str = None) -> dict:
    """
    运行 AI Investment Agent
    
    Args:
        ticker: 股票代码
        query:  自然语言查询（可选）
    """
    if query is None:
        query = f"Analyze {ticker} and provide a comprehensive investment recommendation."
    
    print(f"\n{'='*60}")
    print(f"分析股票: {ticker}")
    print(f"查询: {query}")
    print('='*60)
    
    initial_state = {
        "messages":     [HumanMessage(content=f"Analyze {ticker}. {query}")],
        "ticker":       ticker,
        "tool_results": {},
        "final_signal": "",
        "iterations":   0
    }
    
    # 流式执行，显示每步
    final_state = None
    for step in agent_app.stream(initial_state):
        node_name = list(step.keys())[0]
        node_data = step[node_name]
        
        if node_name == "agent":
            last_msg = node_data["messages"][-1]
            if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    print(f"\n🔧 Tool Call: {tc['name']}({tc['args']})")
            else:
                print(f"\n💬 Final Response:\n{last_msg.content}")
        
        elif node_name == "tools":
            for msg in node_data["messages"]:
                print(f"📊 Tool Result: {str(msg.content)[:200]}...")
        
        final_state = step
    
    # 提取最终回复
    all_messages = []
    for step_data in agent_app.stream(initial_state):
        pass
    
    return final_state

# 测试运行
result = run_agent("NVDA", "Should I buy NVDA given the current AI market trends?")


分析股票: NVDA
查询: Should I buy NVDA given the current AI market trends?

🔧 Tool Call: PriceTool({'ticker': 'NVDA'})

🔧 Tool Call: SentimentTool({'ticker': 'NVDA'})

🔧 Tool Call: QuantTool({'ticker': 'NVDA'})
📊 Tool Result: {"error": "Too Many Requests. Rate limited. Try after a while.", "ticker": "NVDA"}...
📊 Tool Result: {"ticker": "NVDA", "total_articles": 109, "avg_sentiment_score": 0.174, "positive_ratio": 0.367, "negative_ratio": 0.193, "overall_signal": "BULLISH", "sample_headlines": ["Intel's 18A-P chips are in ...
📊 Tool Result: {"ticker": "NVDA", "model": "XGBoost", "prediction": "UP", "up_probability": 0.589, "confidence_interval": [0.489, 0.689], "signal": "BUY", "top_shap_features": [{"feature": "BB_width", "mean_abs_shap...

🔧 Tool Call: PriceTool({'ticker': 'NVDA'})
📊 Tool Result: {"error": "Too Many Requests. Rate limited. Try after a while.", "ticker": "NVDA"}...

🔧 Tool Call: RAGTool({'query': 'NVDA earnings growth AI chip demand', 'ticker': 'NVDA'})


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📊 Tool Result: {"query": "NVDA earnings growth AI chip demand", "ticker_filter": "NVDA", "retrieved_count": 5, "results": [{"headline": "NVDA Reclaims $5 Trillion Market Cap After Best Day In Two Weeks: Retail Trade...

💬 Final Response:
**Final Signal:** BUY  
**Confidence Level:** MEDIUM  

**Key Reasoning:**  
The sentiment analysis for NVDA is bullish with an average sentiment score of 0.174 and a positive news ratio of 36.7%. The machine learning model predicts an upward trend with a 58.9% probability, supported by technical indicators like Bollinger Band width and recent returns. Recent news highlights NVDA's strong market position and strategic financial moves, such as a significant bond sale to fuel AI growth, which supports a positive outlook.

**Risk Factors:**  
1. Market Volatility: The AI chip market is highly competitive, and any technological advancements by competitors could impact NVDA's market share.
2. Economic Conditions: Broader economic downturns or changes in int

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
def run_conversation(turns: list) -> None:
    """
    多轮对话：支持历史上下文
    """
    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    
    for user_input in turns:
        print(f"\n{'='*60}")
        print(f"👤 User: {user_input}")
        print('='*60)
        
        messages.append(HumanMessage(content=user_input))
        
        state = {
            "messages":     messages,
            "ticker":       "",
            "tool_results": {},
            "final_signal": "",
            "iterations":   0
        }
        
        final_response = ""
        for step in agent_app.stream(state):
            node_name = list(step.keys())[0]
            node_data = step[node_name]
            
            if node_name == "agent":
                last_msg = node_data["messages"][-1]
                if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                    for tc in last_msg.tool_calls:
                        print(f"  🔧 {tc['name']}({list(tc['args'].values())})")
                else:
                    final_response = last_msg.content
                    print(f"\n🤖 Agent: {final_response}")
        
        messages.append(AIMessage(content=final_response))
        
        # 保存到 MemoryStore
        if final_response:
            memory.save(
                ticker="multi",
                signal="N/A",
                reasoning=final_response[:500],
                tool_data={}
            )

# 多轮对话测试
run_conversation([
    "Compare NVDA and TSLA. Which one has better momentum right now?",
    "What does the sentiment analysis say about NVDA specifically?",
    "Based on the ML model, what's the prediction for TSLA tomorrow?"
])


👤 User: Compare NVDA and TSLA. Which one has better momentum right now?
  🔧 PriceTool(['NVDA'])
  🔧 PriceTool(['TSLA'])

🤖 Agent: It seems that I'm currently unable to retrieve the latest price and technical indicators for NVDA and TSLA due to rate limiting. Please try again later.

👤 User: What does the sentiment analysis say about NVDA specifically?
  🔧 SentimentTool(['NVDA'])

🤖 Agent: The sentiment analysis for NVDA shows a bullish overall signal with an average sentiment score of 0.174. Among the articles analyzed, 36.7% are positive, while 19.3% are negative. This indicates a generally positive sentiment in the news surrounding NVDA.

👤 User: Based on the ML model, what's the prediction for TSLA tomorrow?
  🔧 QuantTool(['TSLA'])

🤖 Agent: The ML model prediction for TSLA tomorrow is "DOWN" with a probability of 32.8% for an upward movement. The confidence interval for this prediction is between 22.8% and 42.8%. The model suggests a "SELL" signal, with key features influencing th

In [11]:
# 测试历史查询
def query_history(ticker: str = None):
    history = memory.get_history(ticker=ticker, limit=5)
    print(f"\n=== 推荐历史 {'(' + ticker + ')' if ticker else '(全部)'} ===")
    for h in history:
        print(f"[{h['timestamp'][:19]}] {h['ticker']} → {h['signal']}")
        print(f"  {h['reasoning'][:100]}...")
        print()

query_history()


=== 推荐历史 (全部) ===
[2026-06-29T18:37:14] multi → N/A
  The ML model prediction for TSLA tomorrow is "DOWN" with a probability of 32.8% for an upward moveme...

[2026-06-29T18:37:11] multi → N/A
  The sentiment analysis for NVDA shows a bullish overall signal with an average sentiment score of 0....

[2026-06-29T18:37:09] multi → N/A
  It seems that I'm currently unable to retrieve the latest price and technical indicators for NVDA an...

[2026-06-29T18:21:09] conversation → BUY
  Final Signal: HOLD

Confidence: LOW

Reasoning:
- I would not buy TSLA aggressively here based on th...

[2026-06-29T18:20:44] conversation → HOLD
  Final Signal: HOLD

Confidence: LOW

Reasoning:
- I view NVDA as stronger than TSLA mainly on busine...



In [12]:
# 验证 LangSmith tracing 是否正常
from langsmith import Client

try:
    client = Client()
    projects = list(client.list_projects())
    print("LangSmith 连接成功")
    print(f"项目列表: {[p.name for p in projects[:5]]}")
    print(f"\n去 https://smith.langchain.com 查看 '{os.getenv('LANGCHAIN_PROJECT')}' 项目的 trace")
except Exception as e:
    print(f"LangSmith 连接失败: {e}")
    print("检查 LANGCHAIN_API_KEY 是否正确")

LangSmith 连接成功
项目列表: ['ai-investment-agent']

去 https://smith.langchain.com 查看 'ai-investment-agent' 项目的 trace


In [13]:
# 测试错误回退机制
print("=== 测试错误回退 ===")

# 测试不存在的股票
result_invalid = run_agent("INVALID_TICKER_XYZ")
print("\n错误回退测试完成")

# 测试正常股票
print("\n=== 正常股票测试 ===")
for ticker in ["AAPL", "GOOGL"]:
    result = run_agent(ticker)
    time.sleep(2)  # 避免 API rate limit

=== 测试错误回退 ===

分析股票: INVALID_TICKER_XYZ
查询: Analyze INVALID_TICKER_XYZ and provide a comprehensive investment recommendation.

💬 Final Response:
It seems that "INVALID_TICKER_XYZ" is not a valid stock ticker. Please provide a valid stock ticker symbol, such as NVDA, AAPL, or TSLA, so I can perform the analysis and provide an investment recommendation.

错误回退测试完成

=== 正常股票测试 ===

分析股票: AAPL
查询: Analyze AAPL and provide a comprehensive investment recommendation.

🔧 Tool Call: PriceTool({'ticker': 'AAPL'})

🔧 Tool Call: SentimentTool({'ticker': 'AAPL'})

🔧 Tool Call: QuantTool({'ticker': 'AAPL'})
📊 Tool Result: {"error": "Too Many Requests. Rate limited. Try after a while.", "ticker": "AAPL"}...
📊 Tool Result: {"ticker": "AAPL", "total_articles": 110, "avg_sentiment_score": 0.273, "positive_ratio": 0.391, "negative_ratio": 0.118, "overall_signal": "BULLISH", "sample_headlines": ["LGBTQ consumers cut spendin...
📊 Tool Result: {"ticker": "AAPL", "model": "XGBoost", "prediction": "UP", "up_p

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📊 Tool Result: {"query": "GOOGL earnings growth AI competition", "ticker_filter": "GOOGL", "retrieved_count": 5, "results": [{"headline": "Michael Saylor Pushes Back On Dilution Concerns, Says Strategy's Stock Sales...

💬 Final Response:
**Final Signal:** HOLD  
**Confidence Level:** MEDIUM  

**Key Reasoning:**  
The sentiment analysis for GOOGL shows a neutral overall signal with a slight positive sentiment score of 0.027, indicating a balanced view in the market. The QuantTool's prediction suggests a slight upward movement with a probability of 50.7%, but the confidence interval is wide, indicating uncertainty. The SHAP analysis highlights technical indicators like Bollinger Band width and recent returns as influential factors. Recent news highlights competition and regulatory challenges, such as the UK ordering Google to improve search transparency, which could impact future performance.

**Risk Factors:**  
1. Regulatory pressures, particularly in Europe, could impact Google's ope

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
# 基础评估：工具调用准确率
test_cases = [
    {"ticker": "NVDA", "query": "Analyze NVDA"},
    {"ticker": "TSLA", "query": "Should I buy TSLA?"},
    {"ticker": "AAPL", "query": "What is the outlook for AAPL?"},
]

eval_results = []

for case in test_cases:
    try:
        state = {
            "messages":     [HumanMessage(content=f"{case['query']} Ticker: {case['ticker']}")],
            "ticker":       case["ticker"],
            "tool_results": {},
            "final_signal": "",
            "iterations":   0
        }
        
        tools_called = []
        final_response = ""
        
        for step in agent_app.stream(state):
            node_name = list(step.keys())[0]
            node_data = step[node_name]
            if node_name == "agent":
                last_msg = node_data["messages"][-1]
                if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                    tools_called.extend([tc["name"] for tc in last_msg.tool_calls])
                else:
                    final_response = last_msg.content
        
        expected_tools = {"PriceTool", "SentimentTool", "QuantTool"}
        called_set     = set(tools_called)
        tool_accuracy  = len(expected_tools & called_set) / len(expected_tools)
        
        eval_results.append({
            "ticker":        case["ticker"],
            "tools_called":  tools_called,
            "tool_accuracy": tool_accuracy,
            "has_signal":    any(s in final_response.upper()
                                 for s in ["BUY", "SELL", "HOLD"]),
            "response_len":  len(final_response)
        })
        
        print(f"{case['ticker']}: tools={tools_called}, accuracy={tool_accuracy:.0%}")
        time.sleep(2)
        
    except Exception as e:
        print(f"{case['ticker']} 评估失败: {e}")

df_eval = pd.DataFrame(eval_results)
print(f"\n=== Agent 评估报告 ===")
print(f"平均工具调用准确率: {df_eval['tool_accuracy'].mean():.1%}")
print(f"信号生成率:         {df_eval['has_signal'].mean():.1%}")
print(df_eval[["ticker", "tools_called", "tool_accuracy", "has_signal"]].to_string(index=False))

df_eval.to_csv("data/models/agent_eval.csv", index=False)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

NVDA: tools=['PriceTool', 'SentimentTool', 'QuantTool', 'PriceTool', 'RAGTool'], accuracy=100%


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

TSLA: tools=['PriceTool', 'SentimentTool', 'QuantTool', 'RAGTool'], accuracy=100%


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

AAPL: tools=['PriceTool', 'SentimentTool', 'QuantTool', 'RAGTool'], accuracy=100%

=== Agent 评估报告 ===
平均工具调用准确率: 100.0%
信号生成率:         100.0%
ticker                                              tools_called  tool_accuracy  has_signal
  NVDA [PriceTool, SentimentTool, QuantTool, PriceTool, RAGTool]            1.0        True
  TSLA            [PriceTool, SentimentTool, QuantTool, RAGTool]            1.0        True
  AAPL            [PriceTool, SentimentTool, QuantTool, RAGTool]            1.0        True
